In [9]:
from typing import Literal, TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class ApprovalState(TypedDict):
    action_details: str
    status: Literal["pending", "approved", "rejected", None]


def approved_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    decision = interrupt({"question": "是否批准此操作？", "details": state["action_details"]})
    print(decision)
    return Command(goto="proceed" if decision else "cancel")


def proceed_node(state: ApprovalState) -> ApprovalState:
    return {"status": "approved"}


def cancel_node(state: ApprovalState) -> ApprovalState:
    return {"status": "rejected"}


builder = StateGraph(ApprovalState)
builder.add_node("approved", approved_node)
builder.add_node("proceed", proceed_node)
builder.add_node("cancel", cancel_node)
builder.add_edge(START, "approved")
builder.add_edge("proceed", END)
builder.add_edge("cancel", END)

graph = builder.compile(MemorySaver())

In [10]:
config = {"configurable": {"thread_id": "test"}}
initial = graph.invoke({"action_details": "V 我 50", "status": "pending"}, config=config)

print(initial.get("__interrupt__", {}))
resumed = graph.invoke(Command(resume=True), config=config)
print(resumed)

[Interrupt(value={'question': '是否批准此操作？', 'details': 'V 我 50'}, id='c3b0d4aea30c32f04c7135bc9917a01b')]
True
{'action_details': 'V 我 50', 'status': 'approved'}


In [12]:
config = {"configurable": {"thread_id": "test"}}
initial = graph.invoke({"action_details": "V 我 50", "status": "pending"}, config=config)

print(initial.get("__interrupt__", {}))
resumed = graph.invoke(Command(resume=False), config=config)
print(resumed)

[Interrupt(value={'question': '是否批准此操作？', 'details': 'V 我 50'}, id='88b350f14525e6d7fb16774d2c36a80f')]
False
{'action_details': 'V 我 50', 'status': 'rejected'}


In [14]:
config = {"configurable": {"thread_id": "test"}}
initial = graph.invoke({"action_details": "V 我 50", "status": "pending"}, config=config)

print(initial.get("__interrupt__", {}))
resumed = graph.invoke(Command(resume=input("1:Apply\n2:Cancel\n") == "1"), config=config)
print(resumed)

[Interrupt(value={'question': '是否批准此操作？', 'details': 'V 我 50'}, id='40636c0aed0fb11de36a20bff7600e34')]
False
{'action_details': 'V 我 50', 'status': 'rejected'}
